In [1]:
# ===== CELL 0: setup =====
import os, sys, json, glob, random
from pathlib import Path
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0))

WORK = "/kaggle/working/reid_v4_with_cam1"
os.makedirs(WORK, exist_ok=True)

hits = glob.glob("/kaggle/input/**/split.json", recursive=True)
assert hits, "split.json nahi mila -> phase4_Re_id ka notebook output attach karo"
OLD_SPLIT = hits[0]
print("\npurana split :", OLD_SPLIT)

ROOTS = []
for dirpath, dirnames, _ in os.walk("/kaggle/input"):
    if any(d.lower().startswith("cam_") for d in dirnames):
        ROOTS.append(dirpath)
ROOTS = sorted(set(ROOTS))
print("\ncrop roots mile:", len(ROOTS))
for r in ROOTS:
    n = len([d for d in os.listdir(r) if d.lower().startswith("cam_")])
    print(f"  {r}  ({n} chunks)")

assert len(ROOTS) >= 2, "sirf ek root mila -> cam12-extra-crops attach karo"

torch: 2.11.0+cu128 | cuda: True
gpu  : Tesla T4

purana split : /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/split.json

crop roots mile: 3
  /kaggle/input/datasets/payal1238gupta/cam-12-chunk-0-12  (15 chunks)
  /kaggle/input/datasets/payal1238gupta/cam12-extra-crops  (5 chunks)
  /kaggle/input/notebooks/payal1238gupta/phase5-multicam-crops/multicam_crops  (3 chunks)


In [2]:
# ===== CELL 1: multi-root scan + split (Cam_1 ab TRAINING me, Cam_3 = unseen test) =====
import re
from collections import Counter
random.seed(42)
MIN_IMG   = 5
HOLDOUT   = "cam_3_"        # ye camera training me NAHI jayega -> unseen test
HOLD_NAME = HOLDOUT.strip("_").replace("cam", "Cam")   # "Cam_3"
SKIP_CAMS = ["cam_4_"]      # khaali camera (deewar dikhta hai)

def cam_of(name):
    m = re.search(r"cam_\d+", name.lower())
    return m.group(0) if m else "unknown"

old = json.load(open(OLD_SPLIT))
old_train_keys = set(old["train"])
eval_keys = {k for _, k in old["query"]}
print("purane train ids:", len(old_train_keys), "| purane eval ids:", len(eval_keys))

all_ids, holdout_ids = {}, {}
for root in ROOTS:
    tag = Path(root).name[:14]
    for ch in sorted(d for d in os.listdir(root) if d.lower().startswith("cam_")):
        lc = ch.lower()
        if "chunk_032" in ch or any(lc.startswith(s) for s in SKIP_CAMS):
            continue
        for tid in os.listdir(os.path.join(root, ch)):
            p = os.path.join(root, ch, tid)
            if not os.path.isdir(p):
                continue
            imgs = sorted(glob.glob(os.path.join(p, "*.jpg")))
            if len(imgs) < MIN_IMG:
                continue
            plain = f"{ch}/{tid}"
            if lc.startswith(HOLDOUT):
                holdout_ids[plain] = imgs          # sirf test ke liye
                continue
            key = plain if (plain in old_train_keys or plain in eval_keys) else f"{tag}/{plain}"
            all_ids[key] = imgs

assert holdout_ids, f"{HOLD_NAME} crops nahi mile -> phase5_multicam_crops attach karo"
print("kul training-pool identities:", len(all_ids))

train_map = {k: v for k, v in all_ids.items() if k not in eval_keys}
print("\ntrain ids :", len(train_map), f"({sum(len(v) for v in train_map.values())} images)")
print("camera-wise train ids:", dict(Counter(cam_of(k) for k in train_map)))
print(f"{HOLD_NAME} (unseen test) ids:", len(holdout_ids),
      f"({sum(len(v) for v in holdout_ids.values())} images)")

def fix(p):
    if os.path.exists(p):
        return p
    for r in ROOTS:
        cand = os.path.join(r, *p.split("/")[-3:])
        if os.path.exists(cand):
            return cand
    return p

query   = [(fix(p), k) for p, k in old["query"]]
gallery = [(fix(p), k) for p, k in old["gallery"]]
assert os.path.exists(query[0][0]), "eval image paths galat hain"
print("\nCam_12 eval (purana hi): query", len(query), "| gallery", len(gallery))

split = {"train": train_map, "query": query, "gallery": gallery, "holdout_cam1": holdout_ids}
json.dump(split, open(f"{WORK}/split_v3.json", "w"))

assert not (set(train_map) & eval_keys), "LEAK! train mein Cam_12 eval id hai"
assert not any(cam_of(k) == HOLDOUT.rstrip("_") for k in train_map), f"LEAK! train mein {HOLD_NAME} hai"
assert any(cam_of(k) == "cam_1" for k in train_map), "Cam_1 training me nahi aaya!"
print(f"leak check: OK (Cam_12 eval + {HOLD_NAME} training se bahar, Cam_1 training me)")


purane train ids: 236 | purane eval ids: 58
kul training-pool identities: 908

train ids : 850 (61445 images)
camera-wise train ids: {'cam_12': 699, 'cam_1': 42, 'cam_8': 109}
Cam_3 (unseen test) ids: 23 (580 images)

Cam_12 eval (purana hi): query 58 | gallery 4924
leak check: OK (Cam_12 eval + Cam_3 training se bahar, Cam_1 training me)


In [3]:
# ===== CELL 2: dataset + PK sampler =====
import torch, random
import numpy as np
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image

train_map  = split["train"]
train_keys = sorted(train_map)
key2label  = {k: i for i, k in enumerate(train_keys)}
NUM_CLASSES = len(train_keys)

H, W = 256, 128
P, K = 8, 4                      # 8 identities x 4 images = batch 32

train_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.03),
    transforms.RandomGrayscale(p=0.1),
    transforms.Pad(10),
    transforms.RandomCrop((H, W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), value=0),
])
eval_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FlatReID(Dataset):
    """ek item = (image, label) — triplets ab batch ke andar bante hain"""
    def __init__(self, id_map, transform):
        self.items = [(p, key2label[k]) for k, v in id_map.items() for p in v]
        self.tf = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

class PKSampler(Sampler):
    """har batch mein P identities, har identity se K images"""
    def __init__(self, id_map, P, K):
        self.by_label = {}
        idx = 0
        for k, v in id_map.items():
            self.by_label.setdefault(key2label[k], []).extend(range(idx, idx + len(v)))
            idx += len(v)
        self.labels = list(self.by_label)
        self.P, self.K = P, K
        self.num_batches = len(self.labels) // P
    def __len__(self):
        return self.num_batches * self.P * self.K
    def __iter__(self):
        labs = self.labels[:]
        random.shuffle(labs)
        for b in range(self.num_batches):
            for lab in labs[b*self.P:(b+1)*self.P]:
                pool = self.by_label[lab]
                pick = random.sample(pool, self.K) if len(pool) >= self.K else \
                       [random.choice(pool) for _ in range(self.K)]
                for i in pick:
                    yield i

class EvalSet(Dataset):
    def __init__(self, pairs, transform):
        self.pairs, self.tf = pairs, transform
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        p, k = self.pairs[i]
        return self.tf(Image.open(p).convert("RGB")), k

train_ds = FlatReID(train_map, train_tf)
train_dl = DataLoader(train_ds, batch_size=P*K, sampler=PKSampler(train_map, P, K),
                      num_workers=4, drop_last=True, pin_memory=True, persistent_workers=True)
query_dl   = DataLoader(EvalSet(split["query"],   eval_tf), batch_size=64, num_workers=2)
gallery_dl = DataLoader(EvalSet(split["gallery"], eval_tf), batch_size=64, num_workers=2)

print("train identities :", NUM_CLASSES)
print("train images     :", len(train_ds))
print("batches/epoch    :", len(train_dl))
print("query / gallery  :", len(split["query"]), "/", len(split["gallery"]))
x, y = next(iter(train_dl))
print("batch:", tuple(x.shape), "| unique labels in batch:", len(set(y.tolist())), "(8 hona chahiye)")

train identities : 850
train images     : 61445
batches/epoch    : 106
query / gallery  : 58 / 4924
batch: (32, 3, 256, 128) | unique labels in batch: 8 (8 hona chahiye)


In [4]:
# ===== CELL 3: model =====
!pip install -q gdown yacs
!rm -rf /tmp/dpr && git clone -q https://github.com/KaiyangZhou/deep-person-reid.git /tmp/dpr
import sys; sys.path.insert(0, "/tmp/dpr")

import torch, torch.nn as nn, torch.nn.functional as F
import torchreid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARCH = "osnet_ain_x1_0"      # naye cameras ke liye better (instance norm)

class ReIDNet(nn.Module):
    def __init__(self, num_classes, arch=ARCH, pretrained=True):
        super().__init__()
        self.net = torchreid.models.build_model(
            name=arch, num_classes=num_classes, loss="triplet", pretrained=pretrained)
    def forward(self, x):
        out = self.net(x)
        if self.training:
            logits, feat = out
            return logits, feat
        return F.normalize(out, dim=1)

model = ReIDNet(NUM_CLASSES).to(device)
print("arch:", ARCH, "| params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M")

x, y = next(iter(train_dl))
model.train()
logits, feat = model(x.to(device))
print("logits:", tuple(logits.shape), "| feat:", tuple(feat.shape))

/tmp/dpr/torchreid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1-CaioD9NaqbHK_kzSMW8VE4_3KcsRjEo
To: /root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 136MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
arch: osnet_ain_x1_0 | params: 2.61M
logits: (32, 850) | feat: (32, 512)


In [5]:
# ===== CELL 4: batch-hard triplet loss =====
class BatchHardTriplet(nn.Module):
    """har anchor ke liye: sabse door positive, sabse paas negative"""
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking = nn.MarginRankingLoss(margin=margin)
    def forward(self, feat, labels):
        f = F.normalize(feat, dim=1)
        dist = torch.cdist(f, f, p=2)                      # (B, B)
        same = labels.unsqueeze(0) == labels.unsqueeze(1)  # (B, B) bool

        d_ap = (dist * same.float()).max(dim=1)[0]         # hardest positive
        d_an = (dist + same.float() * 1e6).min(dim=1)[0]   # hardest negative

        target = torch.ones_like(d_an)
        loss = self.ranking(d_an, d_ap, target)
        frac_active = (d_an - d_ap < 0.3).float().mean()   # kitne triplets abhi bhi mushkil hain
        return loss, d_ap.mean().item(), d_an.mean().item(), frac_active.item()

ce_loss  = nn.CrossEntropyLoss(label_smoothing=0.1)
tri_loss = BatchHardTriplet(margin=0.3)

# quick test
model.train()
logits, feat = model(x.to(device))
l, ap, an, act = tri_loss(feat, y.to(device))
print(f"triplet loss: {l.item():.4f} | d_ap {ap:.3f} | d_an {an:.3f} | active {act*100:.0f}%")
print("CE loss:", ce_loss(logits, y.to(device)).item())

triplet loss: 0.3752 | d_ap 1.175 | d_an 1.100 | active 100%
CE loss: 6.8070173263549805


In [6]:
# ===== CELL 4b: temporal-gap eval -> Cam_12 (val) + unseen camera (HOLDOUT) =====
import os, numpy as np
from torch.utils.data import DataLoader

def frame_no(p):
    try: return int(os.path.splitext(os.path.basename(p))[0])
    except ValueError: return -1

def make_hard(per_id, name):
    q2, g2, skipped = [], [], 0
    for key, paths in per_id.items():
        paths = sorted(paths, key=frame_no)
        n = len(paths)
        if n < 8:
            skipped += 1; continue
        k = max(1, n // 4)
        q2 += [(p, key) for p in paths[:k]]
        g2 += [(p, key) for p in paths[-k:]]
    print(f"{name} hard eval -> ids: {len(per_id)-skipped} | query: {len(q2)} | gallery: {len(g2)} | skipped: {skipped}")
    return (DataLoader(EvalSet(q2, eval_tf), batch_size=64, num_workers=2),
            DataLoader(EvalSet(g2, eval_tf), batch_size=64, num_workers=2))

per_id = {}
for path, key in split["query"] + split["gallery"]:
    per_id.setdefault(key, []).append(path)
q_dl2, g_dl2 = make_hard(per_id, "Cam_12")
q_c1,  g_c1  = make_hard(split["holdout_cam1"], HOLD_NAME)

@torch.no_grad()
def extract(dl, net):
    net.eval()
    fs, ks = [], []
    for imgs, kk in dl:
        fs.append(net(imgs.to(device, non_blocking=True)).float().cpu())
        ks += list(kk)
    return torch.cat(fs), np.array(ks)

def eval_hard(qdl=None, gdl=None, net=None):
    qdl = qdl or q_dl2; gdl = gdl or g_dl2; net = net or model
    qf, qk = extract(qdl, net); gf, gk = extract(gdl, net)
    sim = (qf @ gf.t()).numpy()
    cmcs, aps = [], []
    for i in range(sim.shape[0]):
        o = np.argsort(-sim[i]); m = (gk[o] == qk[i]).astype(np.int32)
        if m.sum() == 0: continue
        cmcs.append(m.cumsum().clip(max=1))
        pos = np.where(m == 1)[0]
        aps.append(float(np.mean([(j+1)/(r+1) for j, r in enumerate(pos)])))
    cmc = np.mean(cmcs, axis=0)
    return {"Rank-1": float(cmc[0]), "Rank-5": float(cmc[min(4, len(cmc)-1)]), "mAP": float(np.mean(aps))}

def eval_cam1(net=None):
    return eval_hard(q_c1, g_c1, net)

BASE_C12 = eval_hard()
BASE_C1  = eval_cam1()
print("baseline Cam_12 (ImageNet init):", {k: round(v*100, 2) for k, v in BASE_C12.items()})
print(f"baseline {HOLD_NAME}  (ImageNet init):", {k: round(v*100, 2) for k, v in BASE_C1.items()})


Cam_12 hard eval -> ids: 55 | query: 1222 | gallery: 1222 | skipped: 3
Cam_3 hard eval -> ids: 18 | query: 135 | gallery: 135 | skipped: 5
baseline Cam_12 (ImageNet init): {'Rank-1': 32.0, 'Rank-5': 42.06, 'mAP': 22.77}
baseline Cam_3  (ImageNet init): {'Rank-1': 25.19, 'Rank-5': 37.78, 'mAP': 31.67}


In [7]:
# ===== CELL 5: training =====
import time, json

EPOCHS, LR, WARMUP = 40, 3e-4, 5
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)

def lr_at(e):
    if e < WARMUP:
        return (e + 1) / WARMUP
    return 0.5 * (1 + np.cos(np.pi * (e - WARMUP) / max(1, EPOCHS - WARMUP)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

best_map, history = -1.0, []
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    tot_ce, tot_tri, tot_act, nb = 0., 0., 0., 0
    for imgs, labels in train_dl:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits, feat = model(imgs)
            l_ce = ce_loss(logits, labels)
            l_tri, _, _, act = tri_loss(feat, labels)
            loss = l_ce + l_tri
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tot_ce += l_ce.item(); tot_tri += l_tri.item(); tot_act += act; nb += 1
    scheduler.step()

    msg = (f"epoch {epoch+1:3d}/{EPOCHS} | ce {tot_ce/nb:.4f} | tri {tot_tri/nb:.4f} "
           f"| active {100*tot_act/nb:.0f}% | lr {scheduler.get_last_lr()[0]:.2e}")
    if (epoch + 1) % 2 == 0 or epoch == EPOCHS - 1:
        r = eval_hard()
        r1 = eval_cam1()
        history.append({"epoch": epoch+1, "cam12": r, "cam1_unseen": r1})
        msg += (f" | C12 R1 {r['Rank-1']*100:5.2f}% mAP {r['mAP']*100:5.2f}%"
                f" | unseen R1 {r1['Rank-1']*100:5.2f}% mAP {r1['mAP']*100:5.2f}%")
        if r["mAP"] > best_map:
            best_map = r["mAP"]
            torch.save({"state_dict": model.state_dict(), "arch": ARCH,
                        "num_classes": NUM_CLASSES, "epoch": epoch+1, "metrics": r},
                       f"{WORK}/reid_v4_best.pt")
            msg += "  <- best"
    print(msg)

print(f"\ndone in {(time.time()-t0)/60:.1f} min | best Cam_12 mAP {best_map*100:.2f}%")
json.dump(history, open(f"{WORK}/history_v4.json", "w"), indent=2)

epoch   1/40 | ce 7.0032 | tri 0.3625 | active 100% | lr 1.20e-04
epoch   2/40 | ce 6.7343 | tri 0.3451 | active 100% | lr 1.80e-04 | C12 R1 30.85% mAP 27.01% | unseen R1 31.85% mAP 32.77%  <- best
epoch   3/40 | ce 6.4430 | tri 0.3262 | active 99% | lr 2.40e-04
epoch   4/40 | ce 6.0930 | tri 0.3044 | active 96% | lr 3.00e-04 | C12 R1 33.96% mAP 33.60% | unseen R1 23.70% mAP 33.95%  <- best
epoch   5/40 | ce 5.7037 | tri 0.2844 | active 93% | lr 3.00e-04
epoch   6/40 | ce 5.2874 | tri 0.2666 | active 91% | lr 2.99e-04 | C12 R1 37.81% mAP 37.37% | unseen R1 20.74% mAP 30.88%  <- best
epoch   7/40 | ce 4.8728 | tri 0.2497 | active 88% | lr 2.98e-04
epoch   8/40 | ce 4.5303 | tri 0.2445 | active 87% | lr 2.95e-04 | C12 R1 45.83% mAP 39.83% | unseen R1 19.26% mAP 31.07%  <- best
epoch   9/40 | ce 4.2128 | tri 0.2272 | active 85% | lr 2.90e-04
epoch  10/40 | ce 3.9315 | tri 0.2141 | active 83% | lr 2.85e-04 | C12 R1 53.85% mAP 46.56% | unseen R1 22.22% mAP 32.60%  <- best
epoch  11/40 | ce 

In [8]:
# ===== CELL 6: final comparison (old Cam_12-only v2 vs new v4 with Cam_1) + BoxMOT export =====
import json, glob

ck = torch.load(f"{WORK}/reid_v4_best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["state_dict"])
NEW_C12, NEW_C1 = eval_hard(), eval_cam1()

OLD_C12 = OLD_C1 = None
old_hits = glob.glob("/kaggle/input/**/reid_v2_best.pt", recursive=True)
if old_hits:
    ck_old = torch.load(old_hits[0], map_location=device, weights_only=False)
    old_model = ReIDNet(ck_old["num_classes"], arch=ck_old["arch"], pretrained=False).to(device)
    old_model.load_state_dict(ck_old["state_dict"])
    OLD_C12, OLD_C1 = eval_hard(net=old_model), eval_cam1(old_model)
else:
    print("NOTE: phase4_training_v2 attach nahi hai -> purane model ka column khaali rahega")

def f(r, k): return "   -   " if r is None else f"{r[k]*100:6.2f}%"
print("best epoch:", ck["epoch"], "| train identities:", NUM_CLASSES)
print(f"\n===== {HOLD_NAME} = UNSEEN camera (training me kabhi nahi dekha) =====")
print("%-8s %14s %16s %16s" % ("metric", "ImageNet init", "old v2 (Cam_12)", "new v4 (+Cam_1)"))
for k in ["Rank-1", "Rank-5", "mAP"]:
    print("%-8s %14s %16s %16s" % (k, f(BASE_C1, k), f(OLD_C1, k), f(NEW_C1, k)))
print("\n===== Cam_12 (trained camera, check ki ye gira to nahi) =====")
print("%-8s %14s %16s %16s" % ("metric", "ImageNet init", "old v2 (Cam_12)", "new v4 (+Cam_1)"))
for k in ["Rank-1", "Rank-5", "mAP"]:
    print("%-8s %14s %16s %16s" % (k, f(BASE_C12, k), f(OLD_C12, k), f(NEW_C12, k)))

json.dump({"unseen_" + HOLD_NAME: {"imagenet": BASE_C1, "old_v2": OLD_C1, "new_v3": NEW_C1},
           "cam12": {"imagenet": BASE_C12, "old_v2": OLD_C12, "new_v3": NEW_C12},
           "train_identities": NUM_CLASSES, "best_epoch": ck["epoch"], "arch": ARCH},
          open(f"{WORK}/eval_v4.json", "w"), indent=2)

# BoxMOT ke liye export (VS Code demo me REID_W = ye file)
sd = {k[len("net."):]: v for k, v in ck["state_dict"].items()
      if k.startswith("net.") and not k.startswith("net.classifier")}
out_name = f"{WORK}/{ARCH}_with_cam1.pt"
torch.save(sd, out_name)
print("\nsaved:", f"{WORK}/eval_v4.json")
print("BoxMOT weights:", out_name, f"({len(sd)} keys)")


best epoch: 38 | train identities: 850

===== Cam_3 = UNSEEN camera (training me kabhi nahi dekha) =====
metric    ImageNet init  old v2 (Cam_12)  new v4 (+Cam_1)
Rank-1           25.19%           28.89%           31.85%
Rank-5           37.78%           40.74%           39.26%
mAP              31.67%           35.64%           36.40%

===== Cam_12 (trained camera, check ki ye gira to nahi) =====
metric    ImageNet init  old v2 (Cam_12)  new v4 (+Cam_1)
Rank-1           32.00%           66.86%           68.99%
Rank-5           42.06%           76.60%           77.17%
mAP              22.77%           59.21%           58.08%

saved: /kaggle/working/reid_v4_with_cam1/eval_v4.json
BoxMOT weights: /kaggle/working/reid_v4_with_cam1/osnet_ain_x1_0_with_cam1.pt (550 keys)
